In [ ]:
from pathlib import Path
import sys
import inspect

REPO = Path("/workspace/ProtoSAM-repro")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import models.grid_proto_fewshot as modulo_alp

assert Path(modulo_alp.__file__).resolve() == (
    REPO / "models/grid_proto_fewshot.py"
).resolve()

print("Código ALP:", modulo_alp.__file__)
print("\n=== Forward ===")
print(inspect.getsource(modulo_alp.FewShotSeg.forward))

print("\n=== Alinhamento de protótipos ===")
if hasattr(modulo_alp.FewShotSeg, "alignLoss"):
    print(inspect.getsource(modulo_alp.FewShotSeg.alignLoss))
else:
    print("Método alignLoss não encontrado.")

In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import torch
import torch.nn.functional as F

from models.vmamba import Backbone_VSSM
from models.grid_proto_fewshot import FewShotSeg, MultiProtoAsConv

SEMENTE = 1234
random.seed(SEMENTE)
np.random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
torch.cuda.manual_seed_all(SEMENTE)

assert torch.cuda.is_available()
assert "modelo_teste_eft" not in globals(), (
    "O modelo de teste já existe. Não execute esta célula novamente "
    "sem liberar esse modelo ou reiniciar o kernel."
)

DIR_EPISODIOS = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "eft_sp_episodios_inspecao_20261011T144630000218Z"
)
CHECKPOINT = Path(
    "/workspace/ProtoSAM-repro/pretrained_model/vmamba_tiny_v2.pth"
)

# 1. Encoder Tiny: configuração usada nos experimentos anteriores.
encoder_teste_eft = Backbone_VSSM(
    out_indices=(3,),
    pretrained=None,
    patch_size=4,
    in_chans=3,
    num_classes=1000,
    depths=[2, 2, 5, 2],
    dims=96,
    ssm_d_state=1,
    ssm_ratio=2.0,
    ssm_dt_rank="auto",
    ssm_act_layer="silu",
    ssm_conv=3,
    ssm_conv_bias=False,
    ssm_drop_rate=0.0,
    ssm_init="v0",
    forward_type="v05_noz",
    mlp_ratio=4.0,
    mlp_act_layer="gelu",
    mlp_drop_rate=0.0,
    drop_path_rate=0.2,
    patch_norm=True,
    norm_layer="ln2d",
    downsample_version="v3",
    patchembed_version="v2",
    use_checkpoint=False,
    posembed=False,
)

checkpoint = torch.load(
    CHECKPOINT, map_location="cpu", weights_only=True
)
pesos = dict(checkpoint["model"])

# Recupera a normalização aprendida; descarta apenas a cabeça.
for sufixo in ("weight", "bias"):
    pesos[f"outnorm3.{sufixo}"] = pesos.pop(
        f"classifier.norm.{sufixo}"
    )
    pesos.pop(f"classifier.head.{sufixo}")

print(encoder_teste_eft.load_state_dict(pesos, strict=True))
del checkpoint, pesos


# 2. Adaptação da extração; forward e alignLoss oficiais preservados.
class FewShotSegTinyTesteEFT(FewShotSeg):
    def get_encoder(self):
        self.encoder = encoder_teste_eft
        self.config["feature_hw"] = [32, 32]

    def get_cls(self):
        self.cls_unit = MultiProtoAsConv(
            proto_grid=[8, 8],
            feature_hw=[32, 32],
            embed_dim=768,
        )

    def get_features(self, imgs_concat):
        imagens = F.interpolate(
            imgs_concat,
            size=(256, 256),
            mode="bilinear",
            align_corners=False,
        )
        features = self.encoder(imagens)[-1]
        assert features.shape[1:] == (768, 8, 8)

        return F.interpolate(
            features,
            size=(32, 32),
            mode="bilinear",
            align_corners=False,
        )


modelo_teste_eft = FewShotSegTinyTesteEFT(
    image_size=256,
    pretrained_path=None,
    cfg={
        "align": True,
        "debug": False,
        "cls_name": "grid_proto",
        "proto_grid_size": 8,
        "feature_hw": [32, 32],
        "which_model": "vmamba_tiny",
        "lora": 0,
    },
).to("cuda", dtype=torch.float32)

assert type(modelo_teste_eft).forward is FewShotSeg.forward
assert type(modelo_teste_eft).alignLoss is FewShotSeg.alignLoss

modelo_teste_eft.train().requires_grad_(True)
modelo_teste_eft.zero_grad(set_to_none=True)


# 3. Primeiro episódio salvo.
config_episodios = json.loads(
    (DIR_EPISODIOS / "config.json").read_text(encoding="utf-8")
)
registro = config_episodios["episodios"][0]

with np.load(DIR_EPISODIOS / registro["arquivo"]) as dados:
    episodio = {chave: dados[chave].copy() for chave in dados.files}

def preparar_rgb_eft(rgb):
    tensor = torch.from_numpy(
        np.ascontiguousarray(rgb)
    ).permute(2, 0, 1).float().unsqueeze(0).cuda()

    # Mesmos valores de normalização usados na avaliação.
    media = tensor.new_tensor([123.675, 116.28, 103.53])[None, :, None, None]
    desvio = tensor.new_tensor([58.395, 57.12, 57.375])[None, :, None, None]
    return (tensor - media) / desvio

support_rgb = preparar_rgb_eft(episodio["support_rgb"])
query_rgb = preparar_rgb_eft(episodio["query_rgb"])
support_fg = torch.from_numpy(
    episodio["support_mask"]
).float().unsqueeze(0).cuda()
query_alvo = torch.from_numpy(
    episodio["query_mask"]
).long().unsqueeze(0).cuda()


# 4. Forward, perda e backward. Sem optimizer.step().
resultado = modelo_teste_eft(
    supp_imgs=[[support_rgb]],
    fore_mask=[[support_fg]],
    back_mask=[[1.0 - support_fg]],
    qry_imgs=[query_rgb],
    isval=False,
    val_wsize=2,
    show_viz=False,
)

logits, perda_alinhamento = resultado[:2]
assert logits.shape == (1, 2, 256, 256)
assert torch.isfinite(logits).all()

# CE sem pesos para este teste de funcionamento.
perda_segmentacao = F.cross_entropy(logits, query_alvo)
perda_total = perda_segmentacao + perda_alinhamento
assert torch.isfinite(perda_total), "Perda não finita."

perda_total.backward()

gradientes = [
    (nome, parametro.grad)
    for nome, parametro in modelo_teste_eft.encoder.named_parameters()
    if parametro.grad is not None
]
assert gradientes, "Nenhum gradiente chegou ao encoder."
assert all(torch.isfinite(g).all().item() for _, g in gradientes), (
    "Há gradientes não finitos."
)
assert any(torch.count_nonzero(g).item() > 0 for _, g in gradientes), (
    "Todos os gradientes são zero."
)

print("GPU:", torch.cuda.get_device_name())
print("Imagem:", registro["imagem"])
print("Escores:", tuple(logits.shape))
print("Perda de segmentação:", float(perda_segmentacao.detach()))
print("Perda de alinhamento:", float(perda_alinhamento.detach()))
print("Perda total:", float(perda_total.detach()))
print("Tensores do encoder com gradiente:", len(gradientes))
print("Gradientes finitos e pelo menos um não nulo: OK")
print("Nenhum passo de atualização dos pesos foi executado.")

# Libera gradientes e referências ao grafo após a inspeção.
modelo_teste_eft.zero_grad(set_to_none=True)
del resultado, logits, perda_segmentacao, perda_alinhamento, perda_total
del gradientes

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import random

import numpy as np
import torch
import torch.nn.functional as F

assert "modelo_teste_eft" in globals(), (
    "O modelo do teste anterior precisa estar carregado."
)

pasta_episodios = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "eft_sp_episodios_inspecao_20261011T144630000218Z"
)
arquivos = sorted(pasta_episodios.glob("episodio_*.npz"))
assert len(arquivos) == 6, f"Esperados 6 episódios; encontrados {len(arquivos)}."

pasta_saida = Path("/workspace/ProtoSAM-repro/reproducao") / (
    "eft_tiny_teste_otimizacao_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
pasta_saida.mkdir(parents=True, exist_ok=False)

config = {
    "finalidade": "Diagnóstico de 12 atualizações; não é treinamento definitivo",
    "encoder": "vmamba_tiny",
    "semente": 1234,
    "episodios": [str(p) for p in arquivos],
    "atualizacoes": 12,
    "episodios_por_atualizacao": 1,
    "otimizador": "AdamW",
    "lr": 1e-5,
    "weight_decay": 0.0,
    "loss": "CE sem pesos + alinhamento oficial, coeficientes 1 e 1",
    "adaptacao": "Todos os parâmetros habilitados, apenas neste diagnóstico",
    "precisao": "float32, sem autocast",
    "pesos_restaurados_ao_final": True,
    "gpu": torch.cuda.get_device_name(),
    "torch": torch.__version__,
}
(pasta_saida / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

modelo = modelo_teste_eft
device = next(modelo.parameters()).device
assert device.type == "cuda"
assert all(p.requires_grad for p in modelo.parameters())

# Cópia independente em RAM: permite desfazer as atualizações do diagnóstico.
estado_inicial = {
    nome: tensor.detach().cpu().clone()
    for nome, tensor in modelo.state_dict().items()
}
modo_anterior = modelo.training

random.seed(1234)
np.random.seed(1234)
torch.manual_seed(1234)
torch.cuda.manual_seed_all(1234)

media = torch.tensor(
    [123.675, 116.28, 103.53], device=device
).view(1, 3, 1, 1)
desvio = torch.tensor(
    [58.395, 57.12, 57.375], device=device
).view(1, 3, 1, 1)

def carregar_episodio_teste(caminho):
    with np.load(caminho, allow_pickle=False) as dados:
        def imagem(chave):
            x = torch.from_numpy(dados[chave].copy())
            x = x.permute(2, 0, 1).unsqueeze(0)
            return (x.to(device=device, dtype=torch.float32) - media) / desvio

        support = imagem("support_rgb")
        query = imagem("query_rgb")
        fg = torch.from_numpy(dados["support_mask"].copy())
        fg = fg.unsqueeze(0).to(device=device, dtype=torch.float32)
        alvo = torch.from_numpy(dados["query_mask"].copy())
        alvo = alvo.unsqueeze(0).to(device=device, dtype=torch.long)

    assert set(torch.unique(fg).tolist()).issubset({0.0, 1.0})
    assert set(torch.unique(alvo).tolist()).issubset({0, 1})
    return support, fg, query, alvo

def executar_teste_curto():
    otimizador = torch.optim.AdamW(
        modelo.parameters(), lr=config["lr"], weight_decay=0.0
    )
    historico = []
    modelo.train()

    try:
        for passo in range(1, config["atualizacoes"] + 1):
            caminho = arquivos[(passo - 1) % len(arquivos)]
            support, fg, query, alvo = carregar_episodio_teste(caminho)
            otimizador.zero_grad(set_to_none=True)

            retorno = modelo(
                [[support]], [[fg]], [[1.0 - fg]], [query],
                isval=False, val_wsize=2, show_viz=False,
            )
            logits, alinhamento = retorno[0], retorno[1]
            ce = F.cross_entropy(logits, alvo)
            total = ce + alinhamento

            assert torch.isfinite(logits).all().item(), "Logits não finitos."
            assert torch.isfinite(total).all().item(), "Perda não finita."
            total.backward()

            com_gradiente = [
                (nome, p) for nome, p in modelo.named_parameters()
                if p.grad is not None
            ]
            assert com_gradiente, "Nenhum parâmetro recebeu gradiente."
            assert all(
                torch.isfinite(p.grad).all().item()
                for _, p in com_gradiente
            ), "Gradiente não finito."

            # Acompanhar um parâmetro com gradiente não nulo.
            escolhido = next(
                (
                    (nome, p) for nome, p in com_gradiente
                    if torch.count_nonzero(p.grad).item() > 0
                ),
                None,
            )
            assert escolhido is not None, "Todos os gradientes são zero."
            nome_monitorado, parametro = escolhido
            antes = parametro.detach().clone()

            otimizador.step()

            assert all(
                torch.isfinite(p).all().item() for p in modelo.parameters()
            ), "Parâmetro não finito após atualização."

            delta = (parametro.detach() - antes).abs().max().item()
            assert delta > 0, "O parâmetro monitorado não mudou."

            linha = {
                "passo": passo,
                "episodio": caminho.name,
                "ce": ce.detach().item(),
                "alinhamento": (
                    alinhamento.detach().item()
                    if torch.is_tensor(alinhamento)
                    else float(alinhamento)
                ),
                "total": total.detach().item(),
                "parametro_monitorado": nome_monitorado,
                "mudanca_maxima": delta,
            }
            historico.append(linha)

            with (pasta_saida / "passos.jsonl").open(
                "a", encoding="utf-8"
            ) as arquivo:
                arquivo.write(json.dumps(linha, ensure_ascii=False) + "\n")

            print(
                f"{passo:02d}/12 | {caminho.name} | "
                f"CE={linha['ce']:.6f} | "
                f"alinhamento={linha['alinhamento']:.6f} | "
                f"total={linha['total']:.6f} | Δpeso={delta:.3e}",
                flush=True,
            )

            del retorno, logits, alinhamento, ce, total, antes

        return historico

    finally:
        modelo.load_state_dict(estado_inicial, strict=True)
        modelo.zero_grad(set_to_none=True)
        modelo.train(modo_anterior)
        print("\nPesos iniciais restaurados.")
        print("Registros:", pasta_saida)

try:
    historico_teste_eft = executar_teste_curto()
finally:
    del estado_inicial